<a href="https://colab.research.google.com/github/hemanta12/FAQ-chatbot/blob/main/FAQ_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/hemanta12/FAQ-chatbot.git
%cd FAQ-chatbot

Cloning into 'FAQ-chatbot'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 50 (delta 16), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 4.73 MiB | 8.12 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/FAQ-chatbot


In [3]:
!pip install pdfplumber --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 96.7 MB/s eta 0:00:00


In [4]:
import pdfplumber

def load_and_chunk_handbook(pdf_path="handbook.pdf", min_chunk_length=50):
    chunks = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text and len(text.strip()) > min_chunk_length:
                chunks.append(text.strip())
    return chunks

chunks = load_and_chunk_handbook()
print(f"Loaded {len(chunks)} chunks")
print("Sample chunk:\n", chunks[7] if chunks else "No chunks found")

Loaded 76 chunks
Sample chunk:
 Calendar of Academic Events
Fall 2025
Undergraduate Main Campus Classes First Bi-Term
Monday, August 25, 2025 Classes Begin
Monday September 1, 2025 Labor Day
Tuesday, September 2, 2025 Last Day to Register for Classes
Monday, October 6, 2025 Last Day to Drop a Class with a grade of W
Wednesday, October 15, 2025 Last day of First Bi Term
Thursday, October 16, 2025 Fall Break
Friday, October 17, 2025 Fall Break
Second Bi-Term
Monday, October 20, 2025 Classes Begin
Tuesday, October 28, 2025 Last Day to Register for Classes
Wednesday, November 26, 2025 Thanksgiving Break
Thursday, November 27, 2025 Thanksgiving Break
Friday, November 28, 2025 Thanksgiving Break
Monday, December 1, 2025 Last Day to Drop a Class with a grade of W Friday,
December 12, 2025 Last day of Second Bi Term
16 Week Main Campus
Monday, August 25, 2025 Classes Begin
Monday, September 1, 2025 Labor Day
Tuesday, September 2, 2025 Last Day to Register for Classes
Thursday, October 16, 2025

In [5]:
!pip install gradio

In [6]:
!pip install sentence-transformers faiss-cpu --no-deps pillow==10.4.0 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 46.5 MB/s eta 0:00:00


In [7]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')

def build_faiss_index(chunks, embedder):
    chunk_embeddings = embedder.encode(chunks, show_progress_bar=True)
    index = faiss.IndexFlatL2(chunk_embeddings.shape[1])
    index.add(np.array(chunk_embeddings))
    return index

faiss_index = build_faiss_index(chunks, embedder)
print(f"FAISS index built with {faiss_index.ntotal} vectors")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

FAISS index built with 76 vectors


In [53]:
def retrieve(query, k=5, relative_margin=0.4, absolute_ceiling=1.25):
    query_vec = embedder.encode([query])
    distances, indices = faiss_index.search(np.array(query_vec), k)
    dists, idxs = distances[0], indices[0]
    best_dist = dists[0]

    if best_dist > absolute_ceiling:
        return []

    return [chunks[i] for dist, i in zip(dists, idxs) if dist <= best_dist + relative_margin]


# test with a real handbook question
test_results = retrieve("What is the parking policy?")
for i, r in enumerate(test_results):
    print(f"--- Result {i+1} (first 200 chars) ---")
    print(r[:200])
    print()

--- Result 1 (first 200 chars) ---
Parking Lot Regulations
There are four types of parking lots:
• Resident: Lots are for residents, directors, and physical plant parking
• Commuter: Lots are for commuters and faculty/staff/visitors
• 

--- Result 2 (first 200 chars) ---
Where You CANNOT Park with a Resident Permit
• Any red-lined or reserved spaces
• No Parking zones
• Fire Lanes
• Commuter Lots (before 5:00 p.m. daily)
• FSV Lots (at any time)
• General Lots (before

--- Result 3 (first 200 chars) ---
R-6: Mahan Hall
R-7: Harth Hall
Vehicle Fines & Disputes
All University parking fines are to be paid at the Office of Student Accounts. Fines will be billed to the account of the student responsible f



In [9]:
if 'generator' not in dir():
    from transformers import pipeline
    generator = pipeline(
        "text-generation",
       model="microsoft/Phi-4-mini-instruct",
         device_map="auto"
    )
    print("Model loaded.")
else:
    print("Model already loaded, skipping.")


config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


model.safetensors.index.json:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 15.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

Model loaded.


In [10]:
print(generator.model)

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(200064, 3072, padding_idx=199999)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=5120, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_feat

In [54]:
def generate_answer(question, retrieved_chunks):
    if not retrieved_chunks:
        return "That's outside the scope of this handbook. Please ask a related question."

    context = "\n\n".join(retrieved_chunks)

    system_prompt = """You are a helpful assistant answering questions about a university student handbook.

Rules:
1. If the context does not contain information that answers the question, respond with EXACTLY this sentence and nothing else: "That's outside the scope of this handbook. Please ask a related question."
2. Never use knowledge outside the provided context, even for well-known facts (geography, history, general trivia, etc.).
3. Only include information directly relevant to the question. Ignore anything else in the context, even if it seems useful. When in doubt, leave it out.
4. Extract SPECIFIC facts, numbers, and rules. Never write a vague summary like "it covers X, Y, and Z."
5. Use bullet points ("-") for multiple distinct facts, up to 5. Use 2-3 plain sentences for a single fact.
6. Never add information not found in the context."""

    user_prompt = f"""Context:
{context}

Question: {question}

Remember: only use parts of the context directly relevant to this question, and follow all rules above exactly."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    result = generator(
        messages,
        max_new_tokens=400,
        do_sample=False,
        max_length=None,
    )
    answer = result[0]['generated_text'][-1]['content'].strip()
    return answer

In [60]:
test_question1 = "What is the parking policy?"
test_answer1 = generate_answer(test_question1, retrieve(test_question1))
print(test_answer1)

test_question2 = "What is the capital of France?"
test_answer2 = generate_answer(test_question2, retrieve(test_question2))
print(test_answer2)



[transformers] Both `max_new_tokens` (=400) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


- There are four types of parking lots: Resident, Commuter, Faulty/Staff/Visitor, and General Lots (G-1 to G-4).
- General lots are for visitors and commuter students only before 5:00 p.m. (Monday–Friday), and all registered vehicles may park after 5:00 p.m. daily.
- Moss residents' spaces in G-1 are always restricted to Moss residents, regardless of time.
- Parking spots are identified with red, white, or blue lines: Red for Faculty/Staff/Visitor, White for Student (Resident or Commuter) & General Use, Blue for Handicap Accessible.
- Parking violations can result in a $20.00 fine or towing at the owner's expense, with escalating fines for repeated violations.
- Vehicles in "No Parking," "Reserved," "Handicapped," "Faculty/Staff/Visitor," "Loading," "Bus Zone/Parking," "24 Hour Enforced," or sidewalk areas may be towed.
- Violations can be paid in full or charged to the student’s account at the Office of Student Accounts.
- Commuting student permits allow parking in white-lined spaces 

In [31]:
import gradio as gr

def answer_question(message, history):
    try:
        retrieved_chunks = retrieve(message)
        answer = generate_answer(message, retrieved_chunks)
        return answer
    except Exception as e:
        return f"Sorry, something went wrong: {e}"

demo = gr.ChatInterface(
    fn=answer_question,
    title="Handbook FAQ Chatbot",
    description="Ask a question about the handbook and get an answer.",
    examples=["What is the parking policy?", "What is the attendance policy?"],
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6c42efb1e97387b957.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
